# Metronome Compass Calibration

A Python port of RNGReporter's HGSS **Seed to Time** verification panel, built for
gathering Metronome-compass calibration data.

Give it a target datetime + delay and a search window, and it enumerates every candidate
seed nearby.  For each seed it reports:

- **Roamer relocation** — where Raikou / Entei / Latios(Latias) move to when the save is
  reloaded (given where they are now).
- **Elm phone-call sequence** — the `P`/`E`/`K` calls you can read off in-game.

Two sections identify the seed two ways: **Section A** from roamer routes + Elm calls
(`a_seed`), **Section B** from the Metronome battle (`b_seed`).  All logic lives in
`utils/calibration_tools.py`.

## Keyboard fixup

The `2` and `w` keys on my keyboard are flaky, so every `input()` prompt below accepts
`\T` for `2` and `\V` for `w` (substituted before the value is used).  It's applied
automatically at each interactive step — ipykernel resets `input` once per cell, so the
library re-installs the fixup at every prompt.  To add pairs, edit `INPUT_SUBS` in
`utils/calibration_tools.py`.


In [5]:
%load_ext autoreload
%autoreload 2
import datetime as dt
from utils.calibration_tools import (
    # Section A -- roamer routes + Elm calls
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section B -- Metronome-compass battle
    generate_candidates_near,
    print_candidates,
    narrow_candidates,
    prompt_magikarp,
    # Persist a run
    save_compass_run,
    # Timer calibration math
    calibrate_timer,
    # Review + apply a model update (deliberate; not automatic on save)
    update_calibration_model,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Section A — Roamer + Elm identification  (→ `a_seed`)

**Configure** your target datetime/delay, the search window, and each roamer's **current**
route (where it is *now*, before the reset) plus whether it's still roaming.  Use `0` for a
current route you don't know or care about.  Variables are `a_`-prefixed so they won't clash
with Section B.

**Identify** (after loading the save, read the roamer map and Elm phone):

1. **Roamer routes** — one number per *roaming* legendary in **R E L** order (e.g. `38 42 11`);
   `.` leaves a roamer unconstrained.  Only roamers marked `present` are expected.
2. If more than one candidate matches, **Elm calls** — type `P`/`E`/`K` as you hear each
   call (matched as a substring, since RNG may advance first); other characters are ignored.
   Type `M` to pick a candidate by number instead.
3. The single surviving row is saved to **`a_seed`** (integer seed is `a_seed["seed"]`).


In [18]:
# --- Section A: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)
a_target_delay   = 681
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset) and whether it30 is still roaming.
a_prev_routes = {"r": 43, "e": 45, "l": 6}
a_present     = {"r": True, "e": True, "l": True}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, present=a_present, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, a_present, display_limit=a_display_limit)
# GOALS: EKP, KPEK, PEKK, EKKP

Observed roamer routes (R E L, space-separated, . = any):  45 34 13



Observed R=45 E=34 L=13  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C0  2025-07-24 14:45:54     679    -2   -1   45  34  13   3  EKPPEKKKPPEPPKE
  0x0C0E02C0  2025-07-24 14:45:55     679    -2   +0   45  34  13   3  PPPKEPPPEEKPKEE
  0x0D0E02C0  2025-07-24 14:45:56     679    -2   +1   45  34  13   3  KEKEEEKPKKPKEKE


Elm calls (type P/E/K as heard; M = pick manually):  ppk


Elm calls so far: PPK
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C0  2025-07-24 14:45:54     679    -2   -1   45  34  13   3  EKPPEKKKPPEPPKE
  0x0C0E02C0  2025-07-24 14:45:55     679    -2   +0   45  34  13   3  PPPKEPPPEEKPKEE


Elm calls (type P/E/K as heard; M = pick manually):  ep


Elm calls so far: PPKEP
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02C0  2025-07-24 14:45:55     679    -2   +0   45  34  13   3  PPPKEPPPEEKPKEE

=== Seed identified: 0x0C0E02C0  2025-07-24 14:45:55  delay=679  R/E/L=45/34/13  Elm=PPPKEPPPEEKPKEE ===


## Section B — Expedition-style Seed Identification  (→ `b_seed`)

Ported from the Metronome Compass Testing notebook.  Generate every candidate seed near a
target `(time, delay)`, each with its precomputed Metronome battle path, then walk the real
battle turn by turn — candidates whose path diverges from what you observe drop out until a
single seed remains, saved as **`b_seed`** (the whole row).

Config uses `b_`-prefixed names so it won't clash with Section A.  The Metronome user's
movepool besides Metronome is `DEFAULT_EXTRA_MOVES` in `utils/calibration_tools.py`; edit it
there if it changes.  (Seeds here use the same year-correct `seed_for` as Section A.)

Magikarp's **level and gender are asked at run time** (they change each battle); the Metronome user's own gender is the stable `b_metronome_user_is_female` config.


In [19]:
# --- Section B: Metronome-compass target ---
# b_target_time     = dt.datetime(2025, 7, 24, 14, 49, 0)
b_timer_delay = 327919
b_target_time = a_target_time + dt.timedelta(milliseconds=b_timer_delay+5000)
b_target_delay    = 20046-25
b_seconds_window  = 2         # +/- X seconds
b_delay_window    = 2000       # +/- Y delays
b_metronome_only  = False     # True = Metronome-only user; False = + DEFAULT_EXTRA_MOVES
b_metronome_user_is_female = True   # the Metronome user's gender (rarely changes)

# Magikarp's level + gender change every run, so prompt for them at execution time.
# opposite_gender is derived relative to the Metronome user's gender above.
b_magikarp_level, b_opposite_gender = prompt_magikarp(b_metronome_user_is_female)

b_candidates = generate_candidates_near(
    b_target_time, b_target_delay, b_seconds_window, b_delay_window,
    magikarp_level=b_magikarp_level, opposite_gender=b_opposite_gender,
    metronome_only=b_metronome_only,
)

# Walk the real battle turn by turn; candidates diverging from what you observe drop out.
b_seed = narrow_candidates(b_candidates, b_magikarp_level, b_opposite_gender,
                           metronome_only=b_metronome_only)


Magikarp level:  7
Magikarp gender (M/F):  F



20005 / 20005 seeds remain -- next is turn 1
        Seed   Delay    dD  predicted turn 1
  0xF60E4E4E   20021    +0  KspM032-           (Horn Drill)
  0xF50E4E4E   20021    +0  KspM204h           (Charm)
  0xF70E4E4E   20021    +0  KspM171h           (Nightmare)
  0xF40E4E4E   20021    +0  KspM376h           (Trump Card)
  0xF80E4E4E   20021    +0  KspM466h           (Ominous Wind)
  0xF60E4E4D   20020    -1  KspM347            (Calm Mind)
  0xF60E4E4F   20022    +1  KspM029h           (Headbutt)
  0xF50E4E4D   20020    -1  KspM052h           (Ember)
  0xF50E4E4F   20022    +1  KspM201            (Sandstorm)
  0xF70E4E4D   20020    -1  KspM175h           (Flail)
  0xF70E4E4F   20022    +1  KspM324h           (Signal Beam)
  0xF40E4E4D   20020    -1  KspM224h           (Megahorn)
  0xF40E4E4F   20022    +1  KspM062h           (Aurora Beam)
  0xF80E4E4D   20020    -1  KspM314h           (Air Cutter)
  0xF80E4E4F   20022    +1  KspM152h           (Crabhammer)
  ... and 19990 more

--- T

  Metronome selected? (move name or M###):  Mud slap
  Hit, crit, or miss? (h/!/-):  h
  Did the effect proc? (y/n):  y



50 / 20005 seeds remain -- next is turn 2
        Seed   Delay    dD  predicted turn 2
  0xF60E4E3D   20004   -17  KspM151            (Acid Armor)
  0xF80E4E73   20058   +37  KspM055h           (Water Gun)
  0xF60E4E7D   20068   +47  KspM152h           (Crabhammer)
  0xF40E4E87   20078   +57  KspM248            (Future Sight)
  0xF40E4D9A   19841  -180  KspM207h           (Swagger)
  0xF60E4D87   19822  -199  KspM260h           (Flatter)
  0xF40E4D3B   19746  -275  KspM195            (Perish Song)
  0xF60E4D31   19736  -285  KspM098h           (Quick Attack)
  0xF80E4D27   19726  -295  KspM001h           (Pound)
  0xF80E4CF0   19671  -350  KspM352h~          (Water Pulse)
  0xF80E4FBF   20390  +369  KspM420!           (Ice Shard)
  0xF60E4FC9   20400  +379  KspM049h           (Sonic Boom)
  0xF40E4FD3   20410  +389  KspM146h           (Dizzy Punch)
  0xF40E4C4E   19509  -512  KspM153h_          (Explosion)
  0xF60E4C44   19499  -522  KspM056-           (Hydro Pump)
  ... and 35 more



  Metronome selected? (move name or M###):  acid armor



2 / 20005 seeds remain -- next is turn 3
        Seed   Delay    dD  predicted turn 3
  0xF60E4E3D   20004   -17  KspM331hhh         (Bullet Seed)
  0xF80E4BDB   19394  -627  KspM189h~          (Mud Slap)

--- Turn 3: answer what happened in the battle ---


  Metronome selected? (move name or M###):  bullet seed
  Hit, crit, or miss? (h/!/-):  h
  Next hit? (h/!/d for done):  h
  Next hit? (h/!/d for done):  h
  Next hit? (h/!/d for done):  d



1 / 20005 seeds remain -- next is turn 4
        Seed   Delay    dD  predicted turn 4
  0xF60E4E3D   20004   -17  KspM161h           (Tri Attack)

Seed identified: 0xF60E4E3D  time=2025-07-24 14:51:27  delay=20004  dD=-17
Full path: KspM189h~ KspM151 KspM331hhh KspM161h KspM136h KspM428- KspM311h KspM229h KspM354h~ KspM283h
Remaining Metronome moves (turn 4+):
  Turn 4: Tri Attack (M161)
  Turn 5: Hi Jump Kick (M136)
  Turn 6: Zen Headbutt (M428)
  Turn 7: Weather Ball (M311)
  Turn 8: Rapid Spin (M229)
  Turn 9: Psycho Boost (M354)
  Turn 10: Endeavor (M283)


## Section C — Save the run  (→ `data/compass_runs.jsonl`)

Records this calibration run — both identified seeds (`a_seed`, `b_seed`) plus the metadata
below — as one JSON line appended to `data/compass_runs.jsonl`.

You're prompted for a **run tag** (e.g. `300s Samwise`, `11000d Work`), the **target timer
delay**, the **target timer calibration**, and free-form **notes**.  Leaving the tag / delay
/ calibration blank re-uses the previous run's value (notes never default).  The record is
pretty-printed and confirmed (`y`/`n`) before it's written.

> **Saving no longer touches the shared calibration model.** The chart/expedition read
> `data/calibration_model.json`, and changing it invalidates a precomputed chart (~1 h to
> rebuild) — you often save test runs *while* charting a target from the current model. Apply
> model changes deliberately in **Section E**.

In [20]:
# --- Section C: append this run to data/compass_runs.jsonl ---
# Saving does NOT change the shared calibration model (that would invalidate a precomputed
# chart).  Apply model changes deliberately in Section E below.
run_record = save_compass_run(a_seed, b_seed)

Run tag [Compass Target 4]:  
Target timer delay [327496]:  
Target timer calibration [0]:  
Notes:  



{
  "saved_at": "2026-09-10T11:19:40",
  "tag": "Compass Target 4",
  "target_timer_delay": 327496,
  "target_timer_calibration": 0,
  "fresh_boot": true,
  "prior_battles": 0,
  "notes": "",
  "a_seed": {
    "seed": 202244800,
    "seed_hex": "0x0C0E02C0",
    "time": "2025-07-24T14:45:55",
    "delay": 679,
    "sec_delta": 0,
    "delay_delta": -2,
    "r_route": 45,
    "e_route": 34,
    "l_route": 13,
    "rng_calls": 3,
    "elm": "PPPKEPPPEEKPKEE"
  },
  "b_seed": {
    "seed": 4128132669,
    "seed_hex": "0xF60E4E3D",
    "time": "2025-07-24T14:51:27",
    "delay": 20004,
    "sec_delta": 0,
    "delay_delta": -17,
    "path_str": "KspM189h~ KspM151 KspM331hhh KspM161h KspM136h KspM428- KspM311h KspM229h KspM354h~ KspM283h"
  }
}



Save this run? (y/n):  y


Saved to data/compass_runs.jsonl
Calibration model NOT updated (run update_calibration_model() to review + apply changes).


## Section D — Timer calibration math  (delay/calibration ↔ seed_b frame)

Fits the collected runs to answer: **given a timer countdown, what `F_b` frame will I hit?**
`M = target_timer_delay + target_timer_calibration` (ms, calibration signed).

**Two frame "rates" that are easy to confuse:**

- **within-run *average* rate** `= (F_b − F_a)/(T_b − T_a)` — the mean frames/sec over a run.
  This genuinely **rises with M** (≈56.8 → 58.3 Hz here): every run starts at frame ~670 in
  the slow post-boot region, and the longer the run, the more that slow start dilutes out,
  pulling the average up toward the ~60 Hz ceiling.
- **`dF_b/dM`** — the slope you actually need to turn a commanded `M` into `F_b`. This is the
  *instantaneous* rate at battle time, which by 3–7 min is already near the ceiling (~59.6 Hz)
  and barely changing. So **`F_b` vs `M` is essentially a straight line** — just with slope
  ~59.6, **not** the average rate.

**The models (previous vs new), all shown in the report for comparison:**

- **`within_rate` (previous)** — a line whose slope is the within-run *average* rate. Using
  the average rate as the `F_b`-vs-`M` slope is the bug that made residuals explode (~160
  frames) as `M` left the centroid, tanking predictions far from the calibrated delays.
- **`linear_m` (NEW, recommended ★)** — fits `F_b` directly against `M` (robust Theil–Sen).
  Correct slope → residuals drop to ~30 frames, and predictions hold across the whole range.
- **`quad_m` (NEW, experimental)** — adds an `M²` term to test the rising-rate curvature
  directly. So far it barely changes the RMS and its curvature is within noise (near the
  ceiling there's little left to bend), so it isn't used by default — worth re-checking as
  the sweep extends toward 10 min.

`model["recommended"]` (= `linear_m`) drives the top-level `predict` / `solve` /
`hit_probability`; each individual model is under `model["models"][name]`.

**Uncertainty is still split two ways:** reducible **mean-uncertainty** (shrinks with more
runs; ~0 near measured delays, grows as you extrapolate) and irreducible **physical jitter**
(`σ ≈ c·√M`) — the latter sets your real hit odds, so use a `tolerance` window that reflects
which frames are actually acceptable.


---

**Option 1 vs option 2 — how to *place* `F_b` (year handling):**

- **Option 1 (current, `linear_m` ★)** — fits `F_b` directly; the intercept `α` swallows
  `F_a` *and* the year term as one constant. Correct only for the year you calibrated in —
  a `F_b` prediction used as a seed frame in year 2025 lands `year−2000 = 25` frames early.
- **Option 2 (`linear_df`)** — fits `dF = F_b − F_a`, then reconstructs
  `F_b = dF(M) + F_a(actual)`, where `F_a(actual)` is the low-16 of your *identified* initial
  seed (which already carries the year). **Year-agnostic by construction**, and old runs stay
  reusable across target years.

The report prints an **apples-to-apples verdict** — the `dF` fit's RMS residual *is* option 2's
`F_b`-placement error (`dF − f_dF = (F_a+dF) − (f_dF+F_a)`), so the two RMS numbers compare
directly. On the current data the two are within noise of each other (`F_a` only wobbles ~14
frames, so removing it barely tightens a ~74-frame residual) — i.e. **option 2 costs no
accuracy**; the choice is about the year handling, not the fit quality.


In [21]:
# --- Section D: fit the collected runs; predict the battle frame from a timer delay ---
model = calibrate_timer()   # reads data/compass_runs.jsonl, prints the model comparison

# === Forward prediction: given a timer delay + calibration, what frame will I land on? ===
d_delay       = 327919    # target_timer_delay (ms)
d_calibration = -0        # target_timer_calibration (ms, signed)
d_Fa          = 706       # initial-seed low16 = key_seed & 0xFFFF (carries the YEAR: true_delay_a + (year-2000))

# --- F_b models (predict F_b directly; intercept folds in F_a + year, so this is the frame
#     ONLY for the calibration year -- shown for comparison) ---
def predict_report(sub, label):
    p = sub["predict"](d_delay, d_calibration)   # range = expected +/- (band + 2*jitter)
    print(f"    [{label:<28}] F_b = {p['expected']:8.1f}   ~95% range "
          f"[{p['lo']:.0f}, {p['hi']:.0f}]   (jitter +/-{p['jitter']:.1f})")

print(f"\nWith delay={d_delay} cal={d_calibration}  (M = {d_delay + d_calibration} ms):")
print("  F_b-direct models (calibration-year frame, for comparison):")
predict_report(model["models"]["within_rate"], "previous within-rate")
predict_report(model["models"]["linear_m"],    "F_b-vs-M line")
if "quad_m" in model["models"]:
    predict_report(model["models"]["quad_m"],  "F_b-vs-M quadratic")

# --- Recommended: dF model (option 2).  Predicts dF; the ACTUAL battle frame is dF + F_a,
#     where F_a carries the year -> year-correct in ANY year, reuses data across years. ---
rec = model["models"][model["recommended"]]
pr  = rec["predict"](d_delay, d_calibration)             # dF distribution
frame = pr["expected"] + d_Fa                            # actual battle-seed low16
print(f"\n  RECOMMENDED ({model['recommended']}, target=dF):")
print(f"    dF = {pr['expected']:.1f}  ->  actual F_b = dF + F_a({d_Fa}) = {frame:.1f}   "
      f"(jitter +/-{pr['jitter']:.1f})   this is the frame the chart/canon use")

# === How likely to land on a specific ACTUAL frame (within a +/- window)? ===
d_target_frame = round(frame)     # default: the predicted actual frame
d_tolerance    = 25               # half-window (frames) counted as a hit
hp = rec["hit_probability"](d_delay, d_calibration, d_target_frame - d_Fa, tolerance=d_tolerance)
print(f"\nP(land on actual F_b={d_target_frame} +/-{d_tolerance}) = {hp['p']*100:.1f}%  "
      f"(expected off by {hp['delta']:+.1f} frames)")

# === (optional) reverse lookup: which delay centers you on a target ACTUAL frame? ===
sol = rec["solve"](d_target_frame - d_Fa, calibration=d_calibration)   # solve in dF space
print(f"\n(reverse) to center on actual F_b={d_target_frame} holding cal={d_calibration}: "
      f"delay = {sol['delay']:.0f} ms")


=== Timer calibration  (41 run(s), 41 timed, 1 excluded as outlier) ===

  F_b as a function of M = delay + calibration (ms)

  within-run avg rate 58.0682 +/- 0.0119 Hz (dF/dt; rises with M as the slow post-boot frames dilute out)

   within-run-rate slope (previous)       [Fb] slope 58.0682 Hz                 RMS residual  279.3 frames
   F_b-vs-M line (NEW)                    [Fb] slope 59.9852 Hz                 RMS residual   72.4 frames
   F_b-vs-M quadratic (NEW, experimental) [Fb] inst rate 59.534->60.988 Hz over M range RMS residual   57.9 frames
  *dF-vs-M line (option 2)                [dF] slope 59.9325 Hz                 RMS residual   76.7 frames
   dF-vs-M quadratic (option 2, experimental) [dF] inst rate 59.502->61.046 Hz over M range RMS residual   57.9 frames

  ( * = recommended; predict/solve/hit_probability use it )

  option 1 vs 2 (F_b-placement RMS):  F_b-fit 72.4  vs  dF-fit 76.7 frames  ->  F_b (option 1) tighter by 6%
    (dF removes the run-to-run F_a wobble

## Section E — Apply the model update  (→ `data/calibration_model.json`)

Section D only *reports* the fit; it doesn't change anything. This cell re-fits from all runs,
shows how each parameter differs from the **currently deployed** `data/calibration_model.json`,
and writes the new model **only after you confirm**.

Run it when you actually want the chart/expedition to adopt the latest calibration — **not**
every time you save a test run. Because the chart reads this artifact, applying a change means
the next `x.precompute_chart()` will rebuild (~1 h), so do it deliberately between charting
sessions.

In [4]:
# --- Section E: review the re-fit vs the deployed model, then write it only if confirmed ---
# Prints an old -> new parameter table and asks before overwriting data/calibration_model.json.
# Applying it means the chart is stale until you re-run x.precompute_chart().
new_model = update_calibration_model()


Proposed calibration-model change (old -> new):
  kind                           line -> line          
  n_runs                           36 -> 37              <-- changed
  n_fit                            35 -> 36              <-- changed
  beta                       0.060006 -> 0.059968        <-- changed
  alpha                        368.03 -> -299.21         <-- changed
  coeffs                           () -> ()            
  jitter_c                    0.12563 -> 0.12329         <-- changed
  jitter_rms                   74.589 -> 76.128          <-- changed
  rtc_offset_seconds           5.2596 -> 5.308           <-- changed
  m_lo                         175000 -> 175000        
  m_hi                         595000 -> 595000        

*** Applying this invalidates the current precomputed chart -- you'll need to re-run precompute_chart (~1 h). ***


Write this model? (y/n):  y


Updated calibration model -> data/calibration_model.json
